In [43]:
import os
import qdrant_client
from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.vector_stores.types import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.core.vector_stores.types import VectorStoreQueryMode
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core import QueryBundle
import sqlite3
from llama_index.core.schema import TextNode

conn = sqlite3.connect("../chunking/knowledge_base.db", check_same_thread=False)
cursor = conn.cursor()


VLLM_API_BASE_URL = os.getenv("VLLM_API_BASE_URL")
QDRANT_URL = os.getenv("QDRANT_URL")
#collectionname = "WAMASRAGNORIASSUNTO"
collectionname = "WAMASSMALLTOBIGWINDOW" 

client = qdrant_client.QdrantClient(url=QDRANT_URL)


embed_model = OpenAIEmbedding(
    api_base=VLLM_API_BASE_URL,
    model_name="BAAI/bge-m3",
    api_key="null",
)
Settings.embed_model = embed_model


vector_store = QdrantVectorStore(
    client=client,
    collection_name=collectionname,
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)


index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

topk = 40
    
retriever_dense = index.as_retriever(
        vector_store_query_mode=VectorStoreQueryMode.DEFAULT, 
        similarity_top_k=topk,
        filters=None
    )
retriever_sparse = index.as_retriever(
        vector_store_query_mode=VectorStoreQueryMode.SPARSE,
        sparse_top_k=topk,
        filters=None
    )

retriever = QueryFusionRetriever(
    retrievers=[retriever_dense, retriever_sparse],
    similarity_top_k=2*topk,
    num_queries=1,          
    mode="simple",                                       
    use_async=False
    )

query_bundle = QueryBundle(
    query_str="come creo una udc?",
    custom_embedding_strs=None
    )

### Reranker qua

In [44]:
import requests

url = os.getenv("RERANKER_API_URL")

def reranka(nodes, query_text, top_n=5):

    nodinuovi = []
    noditesto = []


    if not nodes:
        return []


    for n in nodes:
        if hasattr(n, 'node'): 

            content = n.node.get_content()
        else:

            content = n.get_content()
        noditesto.append(content)

    payload = {
        "model": "BAAI/bge-reranker-v2-m3",
        "query": f"{query_text}",
        "documents": noditesto,
        "top_n": top_n
    }

    headers = {
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        
        
        if response.status_code != 200:
            print(f"Errore API Rerank ({response.status_code}): {response.text}")
            response.raise_for_status()
            
        results = response.json()
        
        
        for item in results.get("results", []):
            original_index = item["index"]
        
            nodinuovi.append(nodes[original_index])

    except Exception as e:
        print(f" Errore durante il reranking: {e}")
        
        return nodes[:top_n]

    return nodinuovi


In [ ]:
import os
from dotenv import load_dotenv
from llama_index.core.schema import TextNode  

load_dotenv()

def retrieve_with_sql_context(query, user_role, expansion_window=1):
    

    nodes = retriever.retrieve(query)
    nodes = reranka(nodes, query, top_n=10) #10 importante
    
    if not nodes:
        return "Nessun risultato trovato."

    nodi_per_rerank = []
    
    
    nodipostrerank = []


    #espando il contesto del nodo
    for node in nodes:
        
        try:
            res = cursor.execute(
                "SELECT COUNT(*) FROM document_chunks WHERE filename = ?", 
                (node.filename,)
            ).fetchone()
            nchunks = res[0] if res else 0
        except Exception as e:
            print(f"Errore SQL count per {node.filename}: {e}")
            continue
        
        chunksingoli = set() 
        
        mid = node.chunk_index 
        next_one = node.chunk_index + 1 
        end = node.chunk_index + 2
            
            
        if mid < nchunks: #succede sempre ma vabe
            chunksingoli.add(mid)
        if next_one < nchunks:
            chunksingoli.add(next_one)
        if end < nchunks:
            chunksingoli.add(end)


        for i in chunksingoli: #che dovrebbero essere sempre 3
            cursor.execute(
                """
                SELECT text_content FROM document_chunks 
                WHERE filename = ? AND chunk_index = ?
                """, 
                (node.filename, i)
            )
            rows = cursor.fetchall()#questa query dovrebbe tornare sempre 1 riga
            
            if not rows:
                continue
            
            testo = rows[0]
            meta = {"filename": node.filename, "chunk_index": i}
            
            nodo = TextNode(
            text=testo,
            metadata=meta
            )
            
            nodipostrerank.append(nodo)

            
            # if not block_text or not block_text.strip(): # controllo inutile
            #     continue


    testoprererank = []
    
    if not nodi_per_rerank:
        return "Nessun contesto valido recuperato dal DB."
    
    testoprererank[0] = nodipostrerank[0].get_content() + "\n" + nodipostrerank[1].get_content() + "\n" + nodipostrerank[2].get_content()
    testoprererank[1] = nodipostrerank[3].get_content() + "\n" + nodipostrerank[4].get_content() + "\n" + nodipostrerank[5].get_content()
    testoprererank[2] = nodipostrerank[6].get_content() + "\n" + nodipostrerank[7].get_content() + "\n" + nodipostrerank[8].get_content()
    testoprererank[3] = nodipostrerank[9].get_content() + "\n" + nodipostrerank[10].get_content() + "\n" + nodipostrerank[11].get_content()
    testoprererank[4] = nodipostrerank[12].get_content() + "\n" + nodipostrerank[13].get_content() + "\n" + nodipostrerank[14].get_content()


    nodi_ordinati = reranka(, query, top_n=5)
    
    testofinale = "\n\n".join([n.get_content() for n in nodi_ordinati])
    
    return testofinale



In [51]:
print(retrieve_with_sql_context("ho un disallineamento logico fisico in un pallet, come faccio?", "user"))

chunk numero 4 del documento UTL_Refresh_Reset_OT.pdf 
2. Quando usare il refresh -reset degli ordini di trasporto
Le funzioni refresh e reset vengono utilizzati principalmente in quattro casi differenti:
1. Quando su WAMAS ho un ordine di trasporto in stato 'Nuovo' o 'Attivo' da tanto tempo, ma il pallet non si muove.
2. Quando  ho  un  pallet  bloccato  in  uno  dei  buffer  sorgenti  delle  baie  di  picking all'interno dell'anello .
3. Se su WAMAS nella pagina MF200 non riesco a vedere la rotta associata ad una certa UDC con ordine di trasporto ' attivo ' .
4. Quando su WAMAS vedo una UDC in una certa locazione  che  però  non  risulta  essere  su Lighthouse e quindi è presente un disallineamento logico tra i due sistemi.
chunk numero 5 del documento UTL_Refresh_Reset_OT.pdf 
3. Come usare il refresh -reset degli ordini di trasporto
Per  attivare  il  refresh  e  il  reset  degli  ordini  di  trasporto  si  accede  alla  pagina  MF200  di  WAMAS. Dopodiché selezionare l'unica opzio